In [1]:
import os
import shutil
import pandas as pd
import numpy as np

In [4]:
path = '/usr/local/data/amarkr/mimic-cxr-jpg-2.0.0.physionet.org/'

In [5]:
#read the metadata file
df = pd.read_csv(path + 'mimic-cxr-2.0.0-metadata.csv')
#read the disease presence file
df_disease = pd.read_csv(path + 'mimic-cxr-2.0.0-negbio.csv')
#read the patient file
df_patient = pd.read_csv(path + 'patients.csv')

In [6]:
df_disease.head()

,subject_id,study_id,Atelectasis,Cardiomegaly,Consolidation,Edema,Enlarged Cardiomediastinum,Fracture,Lung Lesion,Lung Opacity,No Finding,Pleural Effusion,Pleural Other,Pneumonia,Pneumothorax,Support Devices
0,10000032,50414267,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
1,10000032,53189527,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
2,10000032,53911762,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
3,10000032,56699142,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
4,10000764,57375967,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN,NaN


In [8]:
#select the patient id and study id of patients with no findings
df_mimic_healthy = df_disease[df_disease['No Finding'] == 1.0]

In [9]:
df_mimic_healthy.shape

(78777, 16)

In [19]:
#select the patient id and study id of patients with pleural effusion
df_disease_PE = df_disease[df_disease['Pleural Effusion'] == 1.0]

path_images = '/usr/local/data/amarkr/lgcig/data/mimic-cxr/images/'

#loop over the patients with pleural effusion and copy the images to the new folder
for count, (index, row) in enumerate(df_disease_PE.iterrows()):
    subject_id = row['subject_id']
    study_id = row['study_id']
    
    subfolder = str(int(subject_id))[:2]
    
    path_image = os.path.join(path, f'files/p{subfolder}/p{str(int(subject_id))}', f's{str(int(study_id))}')
    path_report = os.path.join(path, f'reports/files/p{subfolder}/p{str(int(subject_id))}', f's{str(int(study_id))}.txt')
    #get all the files in this path
    files = os.listdir(path_image)

    print(f'#:{count}')
    if count==10:
        break
    #loop over the files and reference then in metadata file
    for file in files:
        image_id = file.split('.')[0]
        row_image = df[df['dicom_id'] == image_id]
        view_position = row_image['ViewPosition'].values[0]
        
        #copy the image to the new folder if the view position is PA or AP
        if view_position == 'PA' or view_position == 'AP':
            patient_id = row_image['subject_id'].values[0]
            study_id = row_image['study_id'].values[0]
            path_image = os.path.join(path, f'files/p{subfolder}/p{str(int(patient_id))}', f's{str(int(study_id))}', file)
            path_report = os.path.join(path, f'reports/files/p{subfolder}/p{str(int(patient_id))}', f's{str(int(study_id))}.txt')
            
            path_new_images = os.path.join(path_images, file)
            shutil.copyfile(path_image, path_new_images)
            shutil.copyfile(path_report, path_new_images.replace('images','reports').replace('.jpg', '.txt'))
            
        
    

#:0
#:1
#:2
#:3
#:4
#:5
#:6
#:7
#:8
#:9
#:10
